# OCR Pipeline for Historical Documents
## RenAIssance Project - Google Summer of Code Evaluation

This notebook demonstrates an end-to-end OCR pipeline for printed historical documents combining:
- Image preprocessing with OpenCV
- Text extraction using Tesseract OCR
- LLM-based post-correction
- Evaluation metrics (CER/WER)

## 1. Setup and Imports

In [ ]:
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import matplotlib.pyplot as plt
from jiwer import wer, cer
import os
from pathlib import Path

# Configure Tesseract path if needed (Windows)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

print("Libraries imported successfully")

## 2. Dataset Preparation

Expected structure:
```
data/
├── images/          # Scanned document images
├── pdfs/            # PDF documents
└── ground_truth/    # Ground truth text files
```

In [ ]:
# Create directory structure
os.makedirs('data/images', exist_ok=True)
os.makedirs('data/pdfs', exist_ok=True)
os.makedirs('data/ground_truth', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("Directory structure created")

## 3. Image Preprocessing Functions

In [ ]:
def load_image(image_path):
    """Load image from file path"""
    return cv2.imread(image_path)

def pdf_to_images(pdf_path, output_folder='data/images'):
    """Convert PDF to images"""
    images = convert_from_path(pdf_path, dpi=300)
    image_paths = []
    for i, img in enumerate(images):
        path = f"{output_folder}/page_{i+1}.png"
        img.save(path, 'PNG')
        image_paths.append(path)
    return image_paths

def preprocess_image(image):
    """Apply preprocessing: grayscale, denoising, thresholding"""
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Noise removal
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    
    # Adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY, 11, 2
    )
    
    return thresh

def detect_text_region(image):
    """Detect and extract main text region"""
    # Find contours
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return image
    
    # Get bounding box of largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Add padding
    padding = 20
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(image.shape[1] - x, w + 2*padding)
    h = min(image.shape[0] - y, h + 2*padding)
    
    return image[y:y+h, x:x+w]

def visualize_preprocessing(original, preprocessed):
    """Visualize preprocessing steps"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(preprocessed, cmap='gray')
    axes[1].set_title('Preprocessed')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

print("Preprocessing functions defined")

## 4. OCR Pipeline

In [ ]:
def extract_text_tesseract(image, lang='eng'):
    """Extract text using Tesseract OCR"""
    custom_config = r'--oem 3 --psm 6'
    text = pytesseract.image_to_string(image, lang=lang, config=custom_config)
    return text.strip()

def ocr_pipeline(image_path):
    """Complete OCR pipeline for a single image"""
    # Load image
    image = load_image(image_path)
    
    # Preprocess
    preprocessed = preprocess_image(image)
    
    # Detect text region
    text_region = detect_text_region(preprocessed)
    
    # Extract text
    text = extract_text_tesseract(text_region)
    
    return text, image, preprocessed

print("OCR pipeline functions defined")

## 5. LLM-Based Text Correction

This demonstrates how an LLM can post-process OCR output to fix common errors.

In [ ]:
def simulate_llm_correction(ocr_text):
    """Simulate LLM-based correction with rule-based approach"""
    corrections = {
        r'\bl\b': 'I',  # Common OCR error: l -> I
        r'\b0\b': 'O',  # 0 -> O
        r'rn': 'm',     # rn -> m
        r'vv': 'w',     # vv -> w
    }
    
    corrected = ocr_text
    for pattern, replacement in corrections.items():
        import re
        corrected = re.sub(pattern, replacement, corrected)
    
    return corrected

def llm_correction_openai(ocr_text, api_key=None):
    """Real LLM correction using OpenAI API (optional)"""
    if not api_key:
        return simulate_llm_correction(ocr_text)
    
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        
        prompt = f"""Fix OCR errors in this historical document text. 
Correct common OCR mistakes while preserving the original meaning:

{ocr_text}

Return only the corrected text."""
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM API error: {e}. Using simulated correction.")
        return simulate_llm_correction(ocr_text)

print("LLM correction functions defined")

## 6. Evaluation Metrics

In [ ]:
def calculate_metrics(predicted, ground_truth):
    """Calculate CER and WER"""
    cer_score = cer(ground_truth, predicted)
    wer_score = wer(ground_truth, predicted)
    
    return {
        'CER': cer_score,
        'WER': wer_score,
        'Character Accuracy': 1 - cer_score,
        'Word Accuracy': 1 - wer_score
    }

def display_metrics(metrics_before, metrics_after):
    """Display comparison of metrics"""
    print("\n" + "="*50)
    print("EVALUATION METRICS")
    print("="*50)
    print(f"\n{'Metric':<20} {'Before LLM':<15} {'After LLM':<15} {'Improvement'}")
    print("-"*65)
    
    for key in ['CER', 'WER', 'Character Accuracy', 'Word Accuracy']:
        before = metrics_before[key]
        after = metrics_after[key]
        improvement = after - before
        print(f"{key:<20} {before:<15.4f} {after:<15.4f} {improvement:+.4f}")
    print("="*50 + "\n")

print("Evaluation functions defined")

## 7. Demo: Process Sample Document

In [ ]:
# Example: Create a sample document for testing
def create_sample_document():
    """Create a sample document image for testing"""
    from PIL import Image, ImageDraw, ImageFont
    
    # Create white background
    img = Image.new('RGB', (800, 400), color='white')
    draw = ImageDraw.Draw(img)
    
    # Sample historical text
    text = """In the year 1776, the Declaration of Independence
was signed by the founding fathers. This document
proclaimed the freedom and sovereignty of the
thirteen American colonies."""
    
    # Draw text
    draw.text((50, 50), text, fill='black')
    
    # Save
    img.save('data/images/sample.png')
    
    # Save ground truth
    with open('data/ground_truth/sample.txt', 'w') as f:
        f.write(text.replace('\n', ' '))
    
    return 'data/images/sample.png', text.replace('\n', ' ')

# Create sample
sample_path, ground_truth = create_sample_document()
print(f"Sample document created at: {sample_path}")
print(f"\nGround truth:\n{ground_truth}")

## 8. Run Complete Pipeline

In [ ]:
# Process image
ocr_text, original, preprocessed = ocr_pipeline(sample_path)

print("OCR Output (Raw):")
print("-" * 50)
print(ocr_text)
print("\n")

# Visualize preprocessing
visualize_preprocessing(original, preprocessed)

In [ ]:
# Apply LLM correction
corrected_text = simulate_llm_correction(ocr_text)

print("OCR Output (After LLM Correction):")
print("-" * 50)
print(corrected_text)
print("\n")

## 9. Evaluation Results

In [ ]:
# Calculate metrics
metrics_before = calculate_metrics(ocr_text, ground_truth)
metrics_after = calculate_metrics(corrected_text, ground_truth)

# Display results
display_metrics(metrics_before, metrics_after)

## 10. Batch Processing Function

In [ ]:
def process_batch(image_folder, ground_truth_folder, use_llm=True):
    """Process multiple documents and aggregate metrics"""
    results = []
    
    for img_file in Path(image_folder).glob('*.png'):
        # Get corresponding ground truth
        gt_file = Path(ground_truth_folder) / f"{img_file.stem}.txt"
        if not gt_file.exists():
            continue
        
        with open(gt_file, 'r') as f:
            ground_truth = f.read().strip()
        
        # Process
        ocr_text, _, _ = ocr_pipeline(str(img_file))
        corrected = simulate_llm_correction(ocr_text) if use_llm else ocr_text
        
        # Metrics
        metrics = calculate_metrics(corrected, ground_truth)
        
        results.append({
            'file': img_file.name,
            'metrics': metrics
        })
    
    return results

print("Batch processing function defined")

## 11. Conclusion

This pipeline demonstrates:
- Image preprocessing for OCR optimization
- Text extraction using Tesseract
- LLM-based post-correction
- Quantitative evaluation with CER/WER

### Future Improvements:
1. Fine-tune Tesseract for historical fonts
2. Implement CNN-RNN architecture for better accuracy
3. Use transformer-based models (TrOCR, Donut)
4. Expand LLM correction with context-aware prompts
5. Add layout analysis for complex documents